# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Este notebook genera el mapa interactivo de precios y los rankings de cadenas para un mes dado.

▶️ **Instrucciones**: ejecutá las celdas en orden. Solo necesitás modificar la **Celda 2 — Configuración**.

**Requisitos**: tener los archivos SEPA y maestros en tu Google Drive (ver Celda 2).

In [ ]:
# ── Celda 1: Instalación y setup (ejecutar una sola vez) ──────────────────
import sys, os

# Clonar el repositorio
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git /content/repo
    print('✓ Repositorio clonado')
else:
    !git -C /content/repo pull -q
    print('✓ Repositorio actualizado')

# Instalar dependencias
!pip install -q folium branca openpyxl pyarrow

# Agregar src al path
sys.path.insert(0, '/content/repo/src')
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ── Celda 2: CONFIGURACIÓN — única celda que necesitás modificar ──────────
# ══════════════════════════════════════════════════════════════════════════

# Mes a analizar (formato MMAAAA)
MES_ANALIZAR = '042026'  # ← cambiar acá (042026 = abril 2026)

# Carpeta base en tu Google Drive donde están los archivos SEPA
# Ejemplo: si tus archivos están en Mi unidad > bases_sepa
DRIVE_BASE = '/content/drive/MyDrive/bases_sepa'  # ← cambiar si tu carpeta es distinta

# Rutas a los archivos (se construyen automáticamente desde DRIVE_BASE y MES_ANALIZAR)
# Si tu estructura de carpetas es distinta, podés escribir la ruta completa acá:
CSV_PARTE1         = f'{DRIVE_BASE}/{MES_ANALIZAR}_pais_parte1COMPLETO.csv.gz'
CSV_PARTE2         = f'{DRIVE_BASE}/{MES_ANALIZAR}_pais_parte2COMPLETO.csv.gz'
MAESTRO_PRODUCTOS  = f'{DRIVE_BASE}/maestros/Maestro de Productos Interno.xlsx'
MAESTRO_SUCURSALES = f'{DRIVE_BASE}/maestros/maestro_sucursales_completo.xlsx'

# Carpeta de salida (en tu Drive — se crea automáticamente si no existe)
DRIVE_OUTPUT = f'{DRIVE_BASE}/outputs/{MES_ANALIZAR}'

# ══════════════════════════════════════════════════════════════════════════
print(f'Mes: {MES_ANALIZAR}')
print(f'Base Drive: {DRIVE_BASE}')

In [ ]:
# ── Celda 3: Montar Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Verificar que los archivos existen
from pathlib import Path

archivos = {
    'CSV Parte 1':  CSV_PARTE1,
    'CSV Parte 2':  CSV_PARTE2,
    'Maestro productos':  MAESTRO_PRODUCTOS,
    'Maestro sucursales': MAESTRO_SUCURSALES,
}

ok = True
for nombre, ruta in archivos.items():
    existe = Path(ruta).exists()
    emoji = '✓' if existe else '✗'
    print(f'{emoji} {nombre}: {ruta}')
    if not existe:
        ok = False

if not ok:
    print('\n⚠ Algunos archivos no se encontraron. Revisá las rutas en la Celda 2.')
else:
    print('\n✓ Todos los archivos encontrados. Podés continuar.')

In [ ]:
# ── Celda 4: Cargar y limpiar datos ──────────────────────────────────────
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_products, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

# Directorio con los dos CSV.GZ
csv_dir = Path(CSV_PARTE1).parent

print(f'Cargando datos de {MES_ANALIZAR}...')
frames = []
for chunk in iter_semester_csvgz(csv_dir, ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('No se encontraron datos. Verificá que los CSV.GZ del mes estén en la carpeta correcta.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} observaciones cargadas')
print(f'Meses en el dataset: {sorted(df_long["fecha"].dt.to_period("M").astype(str).unique())}')
print(f'EANs de canasta encontrados: {df_long["ean_norm"].nunique()} de 30')

In [ ]:
# ── Celda 5: Enriquecer con maestros ─────────────────────────────────────
mb = load_master_branches(MAESTRO_SUCURSALES)
df_enriched = enrich_with_branches(df_long, mb)

print(f'Sucursales con datos: {df_enriched[["id_comercio","id_bandera","id_sucursal"]].drop_duplicates().__len__():,}')
if 'provincia' in df_enriched.columns:
    print(f'Provincias: {df_enriched["provincia"].nunique()}')
if 'cadena' in df_enriched.columns:
    print(f'Cadenas: {df_enriched["cadena"].nunique()}')

In [ ]:
# ── Celda 6: Calcular canasta por sucursal ────────────────────────────────
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)

# Filtrar al mes objetivo
mes_fmt = f'{MES_ANALIZAR[2:]}-{MES_ANALIZAR[:2]}'  # 042026 → 2026-04
df_branch_mes = df_branch[df_branch['mes'] == mes_fmt].copy()

if df_branch_mes.empty:
    meses_disp = sorted(df_branch['mes'].unique())
    raise RuntimeError(f'No hay datos para {mes_fmt}. Meses disponibles: {meses_disp}')

n_suc = len(df_branch_mes)
avg   = df_branch_mes['canasta_total'].mean()
print(f'✓ {n_suc:,} sucursales con canasta completa')
print(f'Canasta promedio: ${avg:,.0f}'.replace(',', '.'))

In [ ]:
# ── Celda 7: Estadísticas del mes ────────────────────────────────────────
stats = df_branch_mes['canasta_total'].describe(percentiles=[.05, .25, .5, .75, .95])
cv    = stats['std'] / stats['mean'] * 100

print(f'\n══ ICM-UADE — {mes_fmt} ══')
print(f'Promedio nacional:  ${stats["mean"]:>10,.0f}'.replace(',', '.'))
print(f'Mediana:            ${stats["50%"]:>10,.0f}'.replace(',', '.'))
print(f'Mínimo  (p5):       ${stats["5%"]:>10,.0f}'.replace(',', '.'))
print(f'Máximo  (p95):      ${stats["95%"]:>10,.0f}'.replace(',', '.'))
print(f'Dispersión (CV):    {cv:>10.1f}%')
print(f'Sucursales:         {n_suc:>10,d}')

In [ ]:
# ── Celda 8: Generar mapa interactivo Folium ──────────────────────────────
from sepa.viz.maps import make_branch_map

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_branch_mes.columns]
suc_cols    = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                          'sucursales_nombre','localidad_nombre','barrio_caba']
                              if c in df_enriched.columns]
df_suc_info = df_enriched[suc_cols].drop_duplicates(subset=branch_cols)

output_html = f'/content/mapa_canasta_pais_{MES_ANALIZAR}_filtros.html'
make_branch_map(df_branch_mes, df_suc_info, output_html, mes=mes_fmt)
print(f'\n✓ Mapa generado: {output_html}')

# Mostrar inline en Colab
from IPython.display import IFrame
IFrame(output_html, width='100%', height=600)

In [ ]:
# ── Celda 9: Rankings de cadenas ─────────────────────────────────────────
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_branch_mes, df_suc_info, mes=mes_fmt)
rank_amba = amba_ranking(df_branch_mes, df_suc_info, mes=mes_fmt)

print('\n── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .rename(columns={'canasta_promedio':'canasta ($)','vs_promedio_pct':'vs. prom. (%)'}))

p1 = plot_chain_ranking(rank_nac,  f'/content/ranking_cadenas_nacional_{MES_ANALIZAR}.png',
                         title=f'Ranking Nacional — {mes_fmt}')
ipy_display(Image(str(p1)))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .rename(columns={'canasta_promedio':'canasta ($)','vs_promedio_pct':'vs. prom. (%)'}))
    p2 = plot_chain_ranking(rank_amba, f'/content/ranking_cadenas_amba_{MES_ANALIZAR}.png',
                             title=f'Ranking AMBA — {mes_fmt}')
    ipy_display(Image(str(p2)))

In [ ]:
# ── Celda 10: Ranking por provincia ──────────────────────────────────────
from sepa.pipeline.aggregator import aggregate_by_province
from sepa.analysis.basket import basket_by_province
from sepa.viz.charts import plot_province_ranking

if 'provincia' in df_suc_info.columns:
    df_prov = aggregate_by_province(df_branch_mes, df_suc_info)
    df_prov_rank = basket_by_province(df_prov, mes_fmt)

    print('\n── Ranking Provincial ──')
    display(df_prov_rank[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .rename(columns={'canasta_provincia':'canasta ($)','vs_promedio_pct':'vs. prom. (%)'}))

    p3 = plot_province_ranking(df_prov, f'/content/ranking_provincias_{MES_ANALIZAR}.png', mes=mes_fmt)
    ipy_display(Image(str(p3)))
else:
    print('⚠ Sin datos de provincia (verificar maestro de sucursales)')

In [ ]:
# ── Celda 11 (opcional): Ranking de barrios CABA ─────────────────────────
from sepa.analysis.basket import barrio_ranking_caba

if 'barrio_caba' in df_suc_info.columns:
    df_barrios = barrio_ranking_caba(df_branch_mes, df_suc_info, mes=mes_fmt)
    if not df_barrios.empty:
        print(f'\n── Ranking Barrios CABA — {mes_fmt} ({len(df_barrios)} barrios) ──')
        display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
                .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta ($)',
                                  'vs_promedio_caba_pct':'vs. CABA (%)'}))
    else:
        print('Sin suficientes datos de barrios CABA (mínimo 2 sucursales por barrio)')
else:
    print('Barrios CABA no disponibles (el maestro de sucursales no tiene coordenadas o la provincia CABA no está en los datos)')

In [ ]:
# ── Celda 12: Guardar outputs en Google Drive ─────────────────────────────
import shutil
from pathlib import Path

Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

outputs = [
    f'/content/mapa_canasta_pais_{MES_ANALIZAR}_filtros.html',
    f'/content/ranking_cadenas_nacional_{MES_ANALIZAR}.png',
    f'/content/ranking_cadenas_amba_{MES_ANALIZAR}.png',
    f'/content/ranking_provincias_{MES_ANALIZAR}.png',
]

for src in outputs:
    if Path(src).exists():
        dst = f'{DRIVE_OUTPUT}/{Path(src).name}'
        shutil.copy(src, dst)
        size_mb = Path(dst).stat().st_size / 1_048_576
        print(f'✓ {Path(src).name} → Drive ({size_mb:.1f} MB)')
    else:
        print(f'⚠ No encontrado: {src}')

print(f'\n✓ Outputs guardados en: {DRIVE_OUTPUT}')